## Univariate Analysis

**Information Need:** Understand the distribution of values for a given global, local, or case attribute, where applicable with respect to a given activity.

**Motivation:** An attribute may be associated with an activity type but not populated in all of its occurrences, and its populated values may be distributed unevenly. Examining both the extent of its population and the distribution of its values provides a more detailed understanding of the information available for that activity and may reveal missingness or value patterns that warrant further investigation.

**Approach:** Based on an Activity and Attribute combination, determine how often the attribute is populated and derive the distribution of its available values.

**Output:** The attribute's fill rate and value distribution for the selected activity type.

In [ ]:
import pandas as pd
import pm4py
import plotly.express as px

from IPython.display import display
from ipywidgets import interact

# --- Configuration -----------------------------------------------------------

LOG_PATH = "../../data/Road_Traffic_Fine_Management_Process.xes"

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)

display(event_log.head())

## Pattern execution

In [ ]:
# Available activities
activities = sorted(
    event_log[ACTIVITY]
    .unique()
)

# Candidate attributes, excluding standard columns
excluded_columns = {
    CASE_ID,
    ACTIVITY,
    TIMESTAMP
}

attributes = sorted(
    column
    for column in event_log.columns
    if column not in excluded_columns
    and event_log[column].notna().any()
)

In [ ]:
def analyze_attribute_filling(event_log, activity, attribute):
    """Analyze attribute filling for one activity–attribute combination."""

    values = event_log.loc[
        event_log[ACTIVITY] == activity,
        attribute
    ]

    total_count = len(values)
    filled_count = values.notna().sum()
    missing_count = values.isna().sum()
    distinct_count = values.nunique(dropna=True)

    fill_rate = (
        filled_count / total_count
        if total_count > 0
        else 0
    )

    summary = pd.DataFrame({
        "Activity": [activity],
        "Attribute": [attribute],
        "Occurrences": [total_count],
        "Filled Values": [filled_count],
        "Missing Values": [missing_count],
        "Fill Rate": [fill_rate],
        "Distinct Values": [distinct_count]
    })

    return values, summary

In [ ]:
def compute_categorical_distribution(values):
    """Compute counts and shares for populated categorical values."""

    populated_values = values.dropna()

    distribution = (
        populated_values
        .value_counts()
        .rename_axis("Value")
        .reset_index(name="Count")
    )

    if not distribution.empty:
        distribution["Share"] = (
            distribution["Count"] / distribution["Count"].sum()
        )

    return distribution

In [ ]:
def plot_value_distribution(values, attribute, max_categories=50): #Change max_categories in case you want to see more individual data points
    """Visualize the distribution of populated attribute values."""

    populated_values = values.dropna()

    if populated_values.empty:
        return None

    # Numerical attributes: histogram with marginal box plot
    if pd.api.types.is_numeric_dtype(populated_values):
        distribution_data = populated_values.to_frame(name=attribute)

        fig = px.histogram(
            distribution_data,
            x=attribute,
            marginal="box"
        )

        fig.update_layout(
            title=f"Distribution of '{attribute}'",
            xaxis_title=attribute,
            yaxis_title="Occurrences"
        )

        return fig

    # Non-numerical attributes: frequency bar chart
    distribution = compute_categorical_distribution(populated_values)
    total_categories = len(distribution)
    displayed_distribution = distribution.head(max_categories)

    title = (
        f"Most Frequent Values of '{attribute}'"
        if total_categories > max_categories
        else f"Distribution of '{attribute}'"
    )

    fig = px.bar(
        displayed_distribution,
        x="Value",
        y="Count"
    )

    fig.update_layout(
        title=title,
        xaxis_title=attribute,
        yaxis_title="Occurrences"
    )

    return fig

In [ ]:
@interact(
    activity=activities,
    attribute=attributes
)
def show_attribute_filling(activity, attribute):
    values, summary = analyze_attribute_filling(
        event_log,
        activity,
        attribute
    )

    display(
        summary.style.format({
            "Occurrences": "{:,.0f}",
            "Filled Values": "{:,.0f}",
            "Missing Values": "{:,.0f}",
            "Fill Rate": "{:.1%}",
            "Distinct Values": "{:,.0f}"
        })
    )

    populated_values = values.dropna()

    if populated_values.empty:
        print(
            "No populated values are available for the selected "
            "activity–attribute combination."
        )
        return

    #Print all unique values available: 
    unique_values = values.dropna().unique().tolist()
    try:
        unique_values = sorted(unique_values)
    except TypeError:
        pass
    print(f"\nUnique populated values ({len(unique_values):,}):")
    print(unique_values)

    # For categorical attributes, also display the exact frequency table.
    if not pd.api.types.is_numeric_dtype(populated_values):
        distribution = compute_categorical_distribution(values)

        display(
            distribution.style.format({
                "Count": "{:,.0f}",
                "Share": "{:.1%}"
            })
        )

        if len(distribution) > 20: #change if you want to display more
            print(
                f"The attribute has {len(distribution):,} distinct populated values. "
                f"The chart therefore shows the {20} most frequent values."
            )

    fig = plot_value_distribution(values, attribute)
    fig.show()